# SimWorld Studio

**AI-powered 3D scene generation platform** 闂?chat with DeepSeek, Qwen, or another OpenAI-compatible model to build urban scenes in real time.

## What this notebook does
1. Checks your Colab GPU
2. Downloads the minimal SimWorld binary (~15 GB)
3. Installs SimWorld Studio platform
4. Sets up LLM authentication (DeepSeek or Qwen/DashScope API key)
5. Launches everything and gives you a browser URL

**Run all cells in order.** Total setup time: ~5 minutes.

---

## Cell 1: GPU Check

In [ ]:
"""Verify GPU is available 闂?T4 or better required."""
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU (T4)")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

supported = ["T4", "A100", "V100", "L4", "A10"]
if not any(g in gpu_info for g in supported):
    print(f"WARNING: Unrecognized GPU. Supported: {supported}")
    print("Continuing anyway...")
else:
    print("GPU check passed!")

## Cell 2: Download SimWorld (Minimal Binary)

Downloads a minimal UE Editor binary (~15 GB compressed) with just the assets needed for the Studio demo.

In [ ]:
"""Download and extract the minimal SimWorld binary."""
import os

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
SIMWORLD_ARCHIVE = "/tmp/SimWorld-Studio-Minimal.tar.gz"
DOWNLOAD_URL = "https://huggingface.co/datasets/SimWorld-AI/SimWorld-Studio/resolve/main/SimWorld-Studio-Minimal.tar.gz"

UE_BINARY = os.path.join(SIMWORLD_DIR, "Engine", "Binaries", "Linux", "UnrealEditor")

if os.path.exists(UE_BINARY):
    print(f"SimWorld binary already installed at: {SIMWORLD_DIR}")
else:
    print("Downloading SimWorld minimal binary (~15 GB)...")
    !wget -q --show-progress -O {SIMWORLD_ARCHIVE} {DOWNLOAD_URL}
    print("Extracting...")
    !tar xzf {SIMWORLD_ARCHIVE} -C /content/
    !rm -f {SIMWORLD_ARCHIVE}

    if os.path.exists(UE_BINARY):
        os.chmod(UE_BINARY, 0o755)
        # Also chmod the launch script
        launch_sh = os.path.join(SIMWORLD_DIR, "SimWorld-Studio.sh")
        if os.path.exists(launch_sh):
            os.chmod(launch_sh, 0o755)
        print(f"SimWorld binary installed at: {SIMWORLD_DIR}")
    else:
        print("ERROR: UnrealEditor not found after extraction!")
        !find {SIMWORLD_DIR} -maxdepth 3 -type f -name "UnrealEditor*" | head -5

print(f"\nUE Binary: {UE_BINARY}")
print(f"Project: {os.path.join(SIMWORLD_DIR, 'gym_citynav', 'gym_citynav.uproject')}")

## Cell 3: Install Studio Platform

In [ ]:
"""Install the Studio platform from your public GitHub repo."""
import importlib.util
import os
import pathlib
import shutil
import subprocess

# --- Studio platform (pip package from your public GitHub repo) ---
print("[1/3] Installing SimWorld Studio...")

GITHUB_REPO = "dddaaamax/SimWorld-Studio-main"
GITHUB_BRANCH = "main"  # change this if your DeepSeek changes live on another branch
REPO_URL = f"https://github.com/{GITHUB_REPO}.git"
STUDIO_GIT_URL = f"git+{REPO_URL}@{GITHUB_BRANCH}#subdirectory=packaging"
SRC_DIR = pathlib.Path("/content/SimWorld-Studio-src")
WORKSPACE = pathlib.Path("/content/studio_workspace")

result = subprocess.run(
    ["pip", "install", "--upgrade", "--force-reinstall", "-q", STUDIO_GIT_URL],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"  Installed from public repo: {GITHUB_REPO}@{GITHUB_BRANCH}")
else:
    print("  ERROR: Could not install package from public repo.")
    print(f"  {result.stderr[:1000]}")
    print("  Check that the repo is public, the branch name is correct, and packaging/ exists.")

# Verify installation
studio_installed = shutil.which("simworld-studio") is not None
if studio_installed:
    ver = subprocess.run(["simworld-studio", "version"], capture_output=True, text=True).stdout.strip()
    print(f"  {ver}")
else:
    print("  WARNING: simworld-studio CLI not found after install!")

# --- Node.js (usually pre-installed in Colab) ---
print("[2/3] Checking Node.js...")
if not shutil.which("node"):
    print("  Installing Node.js...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"], check=False)
else:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Node.js {node_ver} found.")

# --- Build and inject the latest frontend dist into the installed Python package ---
print("[3/3] Building latest frontend dist from repo...")
spec = importlib.util.find_spec("simworld_arena")
if not spec or not spec.origin:
    raise RuntimeError("simworld_arena package was not installed correctly")

pkg_dir = pathlib.Path(spec.origin).parent
installed_dist = pkg_dir / "server" / "dist"
installed_index = installed_dist / "index.html"
print(f"  Installed package: {pkg_dir}")

if SRC_DIR.exists():
    shutil.rmtree(SRC_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, REPO_URL, str(SRC_DIR)], check=True)

web_dir = SRC_DIR / "simworld_studio_workspace" / "web"
if not (web_dir / "package.json").exists():
    raise RuntimeError(f"Frontend package.json not found at {web_dir}")

subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=str(web_dir), check=True)
subprocess.run(["npx", "vite", "build", "--mode", "development"], cwd=str(web_dir), check=True)

built_dist = web_dir / "dist"
if not (built_dist / "index.html").exists():
    raise RuntimeError("Vite build finished but dist/index.html was not created")

if installed_dist.exists():
    shutil.rmtree(installed_dist)
shutil.copytree(built_dist, installed_dist)
print(f"  Copied built dist to installed package: {installed_dist}")

# If workspace already exists, patch it too and force launcher workspace refresh.
workspace_dist = WORKSPACE / "web" / "dist"
workspace_dist.parent.mkdir(parents=True, exist_ok=True)
if workspace_dist.exists():
    shutil.rmtree(workspace_dist)
shutil.copytree(installed_dist, workspace_dist)
version_file = WORKSPACE / ".studio_version"
if version_file.exists():
    version_file.unlink()
print(f"  Copied dist to workspace: {workspace_dist}")
print("  Cleared workspace version marker so Cell 5 can resync package files.")

print("\nAll dependencies installed, and frontend dist is ready.")



## Cell 4: Choose LLM Provider

Default is **DeepSeek**. You can switch to **Qwen/DashScope** by changing `provider` to `qwen`.

Your API key stays in this Colab runtime environment and is passed only to the selected model provider.

In [ ]:
"""Configure DeepSeek or Qwen/DashScope for SimWorld Studio."""
import os
from getpass import getpass

# Choose provider: "deepseek" or "qwen".
provider = "deepseek"

if provider == "deepseek":
    os.environ["SIMWORLD_LLM_PROVIDER"] = "deepseek"
    os.environ["SIMWORLD_LLM_MODEL"] = "deepseek-chat"  # or "deepseek-reasoner"
    if not os.environ.get("DEEPSEEK_API_KEY"):
        os.environ["DEEPSEEK_API_KEY"] = getpass("Enter your DeepSeek API key: ").strip()
    print(f"DeepSeek configured: model={os.environ['SIMWORLD_LLM_MODEL']}")
elif provider == "qwen":
    os.environ["SIMWORLD_LLM_PROVIDER"] = "qwen"
    os.environ["SIMWORLD_LLM_MODEL"] = "qwen-plus"  # examples: qwen-plus, qwen-max, qwen-turbo
    if not os.environ.get("DASHSCOPE_API_KEY"):
        os.environ["DASHSCOPE_API_KEY"] = getpass("Enter your DashScope API key: ").strip()
    print(f"Qwen/DashScope configured: model={os.environ['SIMWORLD_LLM_MODEL']}")
else:
    raise ValueError("provider must be 'deepseek' or 'qwen'")

print("LLM credentials are stored only in this Colab runtime's environment.")

## Cell 5: Launch SimWorld + Studio

In [ ]:
"""Start SimWorld + Studio as non-root with NVIDIA Vulkan."""
import os, subprocess, time, requests, shlex

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
WORKSPACE = "/content/studio_workspace"
MCP_PORT = 55559
WEB_PORT = 3002
RUN_USER = "simworld"

print("[1/4] Installing rendering dependencies...")
subprocess.run([
    "apt-get", "install", "-y", "-qq",
    "xvfb", "vulkan-tools", "libvulkan1", "libegl1", "libgles2", "libgl1"
], check=False)

print("[2/4] Preparing non-root runtime user...")
subprocess.run(["id", RUN_USER], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if subprocess.run(["id", RUN_USER], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", RUN_USER], check=True)

subprocess.run(["mkdir", "-p", WORKSPACE, "/content/runtime-simworld"], check=True)
subprocess.run(["chown", "-R", f"{RUN_USER}:{RUN_USER}", WORKSPACE, "/content/runtime-simworld"], check=False)

# Let the non-root UE process access NVIDIA devices.
subprocess.run("chmod a+rw /dev/nvidia* 2>/dev/null || true", shell=True)
subprocess.run("chmod a+rw /dev/dri/renderD* 2>/dev/null || true", shell=True)

print("[3/4] Starting virtual display...")
subprocess.run(["pkill", "-f", "Xvfb"], capture_output=True)
xvfb_proc = subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x720x24"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)

nvidia_icd = "/usr/share/vulkan/icd.d/nvidia_icd.json"
vk_icd = nvidia_icd if os.path.exists(nvidia_icd) else ""

env_parts = [
    "DISPLAY=:99",
    "XDG_RUNTIME_DIR=/content/runtime-simworld",
    "CUDA_VISIBLE_DEVICES=0",
    "NVIDIA_VISIBLE_DEVICES=all",
    "NVIDIA_DRIVER_CAPABILITIES=all,graphics,compute,utility",
]
if vk_icd:
    env_parts.append(f"VK_ICD_FILENAMES={vk_icd}")

# Pass the selected LLM provider into the non-root process.
for key in [
    "SIMWORLD_LLM_PROVIDER", "SIMWORLD_LLM_MODEL", "SIMWORLD_LLM_BASE_URL", "SIMWORLD_LLM_API_KEY",
    "DEEPSEEK_API_KEY", "DASHSCOPE_API_KEY", "QWEN_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY",
]:
    value = os.environ.get(key)
    if value:
        env_parts.append(f"{key}={shlex.quote(value)}")

print("[4/4] Launching SimWorld Studio...")
cmd = (
    " ".join(env_parts)
    + f" simworld-studio start"
    + f" --binary {SIMWORLD_DIR}"
    + f" --mcp-port {MCP_PORT}"
    + f" --port {WEB_PORT}"
    + f" --data-dir {WORKSPACE}"
    + " --map /Game/Maps/Empty"
    + " --skip-auth-check"
)

safe_cmd = cmd
for secret_key in ["DEEPSEEK_API_KEY", "DASHSCOPE_API_KEY", "QWEN_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "SIMWORLD_LLM_API_KEY"]:
    secret = os.environ.get(secret_key)
    if secret:
        safe_cmd = safe_cmd.replace(secret, "***")
print("  Command:", safe_cmd)

studio_log = open("/content/studio.log", "w")
studio_proc = subprocess.Popen(
    ["runuser", "-u", RUN_USER, "--", "bash", "-lc", cmd],
    stdout=studio_log,
    stderr=subprocess.STDOUT,
)

print("  Waiting for backend...")
last_error = None
for i in range(60):
    try:
        r = requests.get(f"http://localhost:{WEB_PORT}/api/health", timeout=5)
        print(f"  Studio backend running! Health: {r.json()}")
        break
    except Exception as exc:
        last_error = exc
        time.sleep(2)
        if i in [10, 20, 30, 40, 50]:
            print(f"  Still waiting... ({i*2}s)")
            print("  Recent /content/studio.log:")
            subprocess.run(["tail", "-40", "/content/studio.log"])
else:
    print("  Backend did not become ready.")
    print(f"  Last health-check error: {last_error}")
    print("  Last 120 lines of /content/studio.log:")
    subprocess.run(["tail", "-120", "/content/studio.log"])
    print("  Last 120 lines of UE log if present:")
    subprocess.run(["bash", "-lc", f"tail -120 {shlex.quote(WORKSPACE + '/logs/ue.log')} 2>/dev/null || true"])

print("\nLaunch command is running in background.")


## Cell 6: Get Your Browser URL

In [ ]:
"""Create public tunnel URLs for Studio and the UE Pixel Streaming server."""
import os, re, select, subprocess, time, urllib.parse

WEB_PORT = globals().get("WEB_PORT", 3002)
CIRRUS_HTTP_PORT = 8585

print("Setting up tunnels...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Kill any existing tunnel.
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)


def start_tunnel(port, label):
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    accumulated = ""
    url = None
    for _ in range(45):
        time.sleep(1)
        try:
            ready, _, _ = select.select([proc.stderr], [], [], 0)
            if ready:
                chunk = os.read(proc.stderr.fileno(), 8192).decode(errors="replace")
                accumulated += chunk
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", accumulated)
                if match:
                    url = match.group(0)
                    break
        except Exception:
            pass
    print(f"  {label}: {url or 'FAILED'}")
    return proc, url, accumulated

# Studio web UI tunnel.
tunnel, public_url, studio_tunnel_log = start_tunnel(WEB_PORT, "Studio")

# Cirrus HTTP/WebSocket tunnel. This is required for Live viewport in Colab.
cirrus_tunnel, public_cirrus_url, cirrus_tunnel_log = start_tunnel(CIRRUS_HTTP_PORT, "Cirrus")

if public_url:
    if public_cirrus_url:
        live_url = public_url + "?cirrusUrl=" + urllib.parse.quote(public_cirrus_url, safe="")
    else:
        live_url = public_url

    print("")
    print(f"{'='*72}")
    print("  SimWorld Studio is live!")
    print("")
    print(f"  Open this URL for Studio + Live viewport:")
    print(f"  {live_url}")
    print("")
    if public_cirrus_url:
        print(f"  Direct Cirrus test URL:")
        print(f"  {public_cirrus_url}")
    else:
        print("  WARNING: Cirrus tunnel failed. Studio may open, but Live viewport will not connect.")
    print(f"{'='*72}")
    print("")
    public_url = live_url
else:
    print("Tunnel failed to start. Trying Colab proxy fallback...")
    try:
        from google.colab.output import eval_js
        proxy_url = eval_js(f"google.colab.kernel.proxyPort({WEB_PORT})")
        cirrus_proxy_url = eval_js(f"google.colab.kernel.proxyPort({CIRRUS_HTTP_PORT})")
        public_url = proxy_url + "?cirrusUrl=" + urllib.parse.quote(cirrus_proxy_url, safe="")
        public_cirrus_url = cirrus_proxy_url
        print(f"Open in browser: {public_url}")
        print(f"Direct Cirrus test URL: {public_cirrus_url}")
    except Exception:
        print("Both tunnel methods failed. Check your connection.")
        public_url = f"http://localhost:{WEB_PORT}"
        public_cirrus_url = f"http://localhost:{CIRRUS_HTTP_PORT}"


## Cell 7: Verify Everything Works

Run this cell to confirm all services are healthy before using the Studio.

In [ ]:
"""Automated verification 闂?all checks must pass before using the Studio."""
import requests, socket, subprocess, os

print("Running verification checks...\n")

results = {}

# 1. GPU
r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True)
results["GPU"] = (r.returncode == 0, r.stdout.strip() if r.returncode == 0 else "Not found")

# 2. SimWorld binary
ue_bin = "/content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor"
results["SimWorld Binary"] = (os.path.exists(ue_bin), ue_bin if os.path.exists(ue_bin) else "Not found")

# 3. SimWorld TCP (MCP port)
mcp_port = globals().get('MCP_PORT', 55559)
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(("127.0.0.1", mcp_port))
    s.close()
    results[f"SimWorld MCP ({mcp_port})"] = (True, "Connected")
except Exception as e:
    results[f"SimWorld MCP ({mcp_port})"] = (False, str(e))

# 4. Studio Backend
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    results["Studio Backend"] = (r.status_code == 200, f"HTTP {r.status_code}")
except Exception as e:
    results["Studio Backend"] = (False, str(e))

# 5. Skills
try:
    r = requests.get("http://localhost:3002/api/skills", timeout=10)
    data = r.json()
    count = len(data) if isinstance(data, list) else 0
    results["Skills"] = (count > 0, f"{count} skills loaded")
except Exception as e:
    results["Skills"] = (False, str(e))

# 6. Assets
try:
    r = requests.get("http://localhost:3002/api/assets", timeout=10)
    results["Assets"] = (r.status_code == 200, "Catalog loaded")
except Exception as e:
    results["Assets"] = (False, str(e))

# 7. LLM provider
provider = (os.environ.get("SIMWORLD_LLM_PROVIDER") or "").lower()
if not provider:
    provider = "deepseek" if os.environ.get("DEEPSEEK_API_KEY") else ("qwen" if os.environ.get("DASHSCOPE_API_KEY") else "not set")
model = os.environ.get("SIMWORLD_LLM_MODEL", "")
if provider == "deepseek":
    auth_ok = bool(os.environ.get("DEEPSEEK_API_KEY"))
    auth_detail = f"DeepSeek / {model or 'deepseek-chat'}" if auth_ok else "NOT SET 闂?run Cell 4"
elif provider == "qwen":
    auth_ok = bool(os.environ.get("DASHSCOPE_API_KEY") or os.environ.get("QWEN_API_KEY"))
    auth_detail = f"Qwen/DashScope / {model or 'qwen-plus'}" if auth_ok else "NOT SET 闂?run Cell 4"
else:
    auth_ok = False
    auth_detail = "NOT SET 闂?run Cell 4"
results["LLM Provider"] = (auth_ok, auth_detail)

# 8. Tunnel
tunnel_url = globals().get('public_url', '')
tunnel_ok = bool(tunnel_url) and tunnel_url.startswith("http")
results["Public URL"] = (tunnel_ok, tunnel_url if tunnel_ok else "Not created")

# Print results
all_ok = True
for name, (ok, detail) in results.items():
    icon = "[PASS]" if ok else "[FAIL]"
    if not ok:
        all_ok = False
    print(f"  {icon} {name}: {detail}")

print()
if all_ok:
    print("ALL CHECKS PASSED! SimWorld Studio is ready.")
    if tunnel_url:
        print(f"Open {tunnel_url} in your browser to start building 3D scenes.")
else:
    print("SOME CHECKS FAILED. Common fixes:")
    print("  - SimWorld MCP: wait 30-60s more, then re-run this cell")
    print("  - LLM Provider: run Cell 4 and enter your DeepSeek or DashScope API key")
    print("  - Tunnel: re-run Cell 6")
    print("  - Studio Backend: !cat /content/studio.log")

## Cell 8: Smoke Test (End-to-End)

Sends a test prompt through the full pipeline: **Chat -> LLM -> MCP Tools -> SimWorld**

In [ ]:
"""End-to-end smoke test 闂?verifies the full pipeline works."""
import requests, json

print("Sending test prompt through the full pipeline...\n")
print("Prompt: 'Set up the environment with a sunny sky, then spawn one small building'\n")

try:
    response = requests.post("http://localhost:3002/api/chat", json={
        "message": "Set up the environment with a sunny sky, then spawn one small residential building (BP_Building_01) at the origin. Then take a screenshot.",
        "sessionId": None,
        "skills": ["building_placement"]
    }, stream=True, timeout=180)

    tool_calls = []
    text_output = []
    screenshots = []
    errors = []
    current_event = None  # Track SSE event type

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode('utf-8')

        # SSE comment lines (heartbeat)
        if decoded.startswith(':'):
            continue

        # Parse SSE event/data pairs
        if decoded.startswith('event: '):
            current_event = decoded[7:]
            continue
        if not decoded.startswith('data: '):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or 'unknown'

        if event_type == 'text':
            delta = data.get('delta', '')
            text_output.append(delta)
            print(delta, end='', flush=True)
        elif event_type == 'tool_start':
            name = data.get('displayName', data.get('name', '?'))
            tool_calls.append(name)
            print(f"\n  >> Tool: {name}", flush=True)
        elif event_type == 'tool_result':
            result = data.get('result', '')[:200]
            is_error = data.get('isError', False)
            if is_error:
                errors.append(result)
                print(f"\n  >> ERROR: {result}", flush=True)
            else:
                print(f"\n  >> Result: {result[:100]}...", flush=True)
        elif event_type == 'screenshot':
            screenshots.append(data.get('filepath', ''))
            print(f"\n  >> Screenshot captured!", flush=True)
        elif event_type == 'done':
            cost = data.get('costUsd')
            if cost:
                print(f"\n\n  Cost: ${cost:.4f}")

    print(f"\n\n{'='*50}")
    print(f"Smoke Test Results:")
    print(f"  Tool calls:  {len(tool_calls)} ({', '.join(tool_calls)})")
    print(f"  Screenshots: {len(screenshots)}")
    print(f"  Errors:      {len(errors)}")

    if len(tool_calls) > 0 and len(errors) == 0:
        print(f"\n  SMOKE TEST PASSED!")
        print(f"  Full pipeline working: Chat -> LLM -> MCP -> SimWorld")
    elif len(tool_calls) > 0:
        print(f"\n  PARTIAL PASS 闂?tools fired but some errors occurred.")
    else:
        print(f"\n  SMOKE TEST FAILED 闂?no tool calls detected.")
        print(f"  Check: API key set? SimWorld running? Logs: /content/studio.log")

except requests.exceptions.Timeout:
    print("\nSmoke test timed out (180s). The selected LLM may be slow to respond.")
    print("The platform may still work 闂?try using the browser UI.")
except Exception as e:
    print(f"\nSmoke test error: {e}")
    print("Check /content/studio.log for backend errors.")

---

## Troubleshooting

| Issue | Fix |
|---|---|
| "No GPU detected" | Runtime -> Change runtime type -> GPU (T4) |
| SimWorld TCP fails | Wait 60s more, re-run Cell 7 |
| Studio backend fails | Check: `!cat /content/studio.log` |
| Tunnel fails | Re-run Cell 6, or use Colab proxy |
| LLM errors | Verify your DeepSeek or DashScope API key in Cell 4 |
| Session expires | Colab free tier = 12h max. Re-run all cells. |

### View Logs
```python
!tail -50 /content/ue.log      # SimWorld logs
!tail -50 /content/studio.log    # Studio backend logs
```

## Cell 9: Full DeepSeek/Qwen Pipeline Artifact Run

Runs the practical end-to-end workflow:

`Prompt -> LLM tool calls -> UE scene -> screenshot -> Scene artifact -> TaskSet artifact -> TrainingRun artifact -> CurriculumRun artifact`

The scene generation step is real when UE/MCP is healthy. Task, training, and curriculum outputs are structured artifacts for research workflow validation; they do not yet replace real NavMesh sampling or RL training.


In [ ]:
"""Run the full DeepSeek/Qwen SimWorld Studio artifact pipeline."""
import json, os, time, uuid, random, textwrap
from pathlib import Path
from datetime import datetime, timezone

import requests

BASE_URL = "http://127.0.0.1:3002"
OUT_ROOT = Path("/content/studio_workspace/pipeline_outputs")
RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
OUT_DIR = OUT_ROOT / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENE_PROMPT = """
Build a compact UE urban training scene: sunny weather, a small paved plaza,
4 low-rise buildings around the edges, 8 trees, 3 scooters, 2 carts, and several
street props. Keep the center area navigable and take a screenshot at the end.
""".strip()


def write_json(name, data):
    path = OUT_DIR / name
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  wrote {path}")
    return str(path)


def check_backend():
    print("[1/5] Checking Studio backend...")
    r = requests.get(f"{BASE_URL}/api/health", timeout=10)
    r.raise_for_status()
    health = r.json()
    print("  health:", health)
    if not health.get("ueConnected"):
        raise RuntimeError("UE/MCP is not connected. Run Cell 5 first and confirm MCP is healthy.")
    return health


def run_scene_generation(prompt):
    print("[2/5] Generating scene through /api/chat...")
    payload = {
        "message": prompt,
        "sessionId": None,
        "skills": ["building_placement", "city_layout", "street_furniture", "weather_mood"],
    }
    response = requests.post(f"{BASE_URL}/api/chat", json=payload, stream=True, timeout=360)
    response.raise_for_status()

    events = []
    text_chunks = []
    tool_calls = []
    screenshots = []
    current_event = None
    session_id = None

    for raw in response.iter_lines(decode_unicode=True):
        if not raw:
            continue
        if raw.startswith(":"):
            continue
        if raw.startswith("event: "):
            current_event = raw[7:].strip()
            continue
        if not raw.startswith("data: "):
            continue
        try:
            data = json.loads(raw[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or data.get("type", "unknown")
        events.append({"event": event_type, "data": data})

        if event_type == "system":
            session_id = data.get("sessionId") or data.get("session_id") or session_id
            print(f"  session: {session_id}, provider={data.get('provider')}, model={data.get('model')}")
        elif event_type == "text":
            delta = data.get("delta") or data.get("content") or ""
            if delta:
                text_chunks.append(delta)
                print(delta, end="", flush=True)
        elif event_type in ("tool_start", "tool_details"):
            name = data.get("displayName") or data.get("name") or "unknown_tool"
            if event_type == "tool_start":
                tool_calls.append(name)
                print(f"\n  tool: {name}")
        elif event_type == "screenshot":
            shot = data.get("filepath")
            if shot:
                screenshots.append(shot)
                print(f"\n  screenshot: {shot}")
        elif event_type == "done":
            shot = data.get("latestScreenshot")
            if shot:
                screenshots.append(shot)
            print("\n  scene generation done")
            break
        elif event_type == "error":
            raise RuntimeError(data.get("message", "Unknown /api/chat error"))

    artifact = {
        "id": f"scene_{uuid.uuid4().hex[:10]}",
        "runId": RUN_ID,
        "type": "Scene",
        "prompt": prompt,
        "sessionId": session_id,
        "provider": os.environ.get("SIMWORLD_LLM_PROVIDER", ""),
        "model": os.environ.get("SIMWORLD_LLM_MODEL", ""),
        "toolCalls": tool_calls,
        "screenshots": screenshots,
        "assistantText": "".join(text_chunks).strip(),
        "eventsPath": "chat_events.json",
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }
    write_json("chat_events.json", events)
    write_json("scene_result.json", artifact)

    try:
        requests.post(f"{BASE_URL}/api/scenes", json={
            "id": artifact["id"],
            "name": "DeepSeek/Qwen Colab Scene",
            "description": "Generated by the Colab full pipeline artifact runner.",
            "prompt": prompt,
            "skills": payload["skills"],
            "sessionId": session_id,
            "actors": [],
        }, timeout=20)
    except Exception as exc:
        print("  warning: scene metadata save failed:", exc)

    return artifact


def create_taskset(scene):
    print("[3/5] Creating TaskSet artifact...")
    random.seed(scene["id"])
    episodes = []
    for idx in range(12):
        sx = random.randint(-8000, 8000)
        sy = random.randint(-8000, 8000)
        gx = random.randint(-8000, 8000)
        gy = random.randint(-8000, 8000)
        distance = ((sx - gx) ** 2 + (sy - gy) ** 2) ** 0.5 / 100.0
        episodes.append({
            "episodeId": f"ep_{idx+1:03d}",
            "taskType": "PointNav",
            "start": [sx, sy, 120],
            "goal": [gx, gy, 120],
            "estimatedPathLengthMeters": round(distance * 1.25, 2),
            "status": "artifact_only_pending_navmesh_validation",
        })
    taskset = {
        "id": f"taskset_{uuid.uuid4().hex[:10]}",
        "type": "TaskSet",
        "sceneId": scene["id"],
        "name": "PointNav_12_artifact",
        "episodeCount": len(episodes),
        "episodes": episodes,
        "validation": {
            "navmeshChecked": False,
            "note": "Generated as a Colab artifact. Add query_navmesh MCP support for real reachability validation.",
        },
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }
    write_json("taskset_pointnav.json", taskset)
    return taskset


def create_training_run(taskset):
    print("[4/5] Creating TrainingRun artifact...")
    metrics = []
    for epoch in range(1, 9):
        metrics.append({
            "epoch": epoch,
            "successRate": round(min(0.18 + epoch * 0.075, 0.86), 3),
            "spl": round(min(0.10 + epoch * 0.065, 0.72), 3),
            "collisionRate": round(max(0.62 - epoch * 0.055, 0.16), 3),
        })
    training = {
        "id": f"train_{uuid.uuid4().hex[:10]}",
        "type": "TrainingRun",
        "taskSetId": taskset["id"],
        "algorithm": "artifact_level_pointnav_baseline",
        "status": "artifact_only_not_real_rl_training",
        "metrics": metrics,
        "note": "This records the intended training run shape. Connect SB3/Habitat/custom rollout executor for real training.",
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }
    write_json("training_run.json", training)
    return training


def create_curriculum_run(scene, taskset, training):
    print("[5/5] Creating CurriculumRun artifact...")
    final = training["metrics"][-1]
    decisions = []
    if final["successRate"] > 0.7:
        decisions.append("increase_obstacle_density")
        decisions.append("expand_start_goal_distance")
    if final["collisionRate"] > 0.2:
        decisions.append("keep_center_corridor_clear")
        decisions.append("move_small_props_away_from_main_paths")
    curriculum = {
        "id": f"curriculum_{uuid.uuid4().hex[:10]}",
        "type": "CurriculumRun",
        "sceneId": scene["id"],
        "taskSetId": taskset["id"],
        "trainingRunId": training["id"],
        "round": 1,
        "masteryGate": {
            "successRateTarget": 0.75,
            "splTarget": 0.55,
            "collisionRateMax": 0.25,
            "passed": final["successRate"] >= 0.75 and final["spl"] >= 0.55 and final["collisionRate"] <= 0.25,
        },
        "adaptationDecisions": decisions,
        "nextScenePrompt": textwrap.dedent(f"""
            Revise scene {scene['id']} for the next curriculum round.
            Keep the center navigable, preserve the urban plaza layout, and apply: {', '.join(decisions)}.
            Take a screenshot after the edits.
        """).strip(),
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }
    write_json("curriculum_run.json", curriculum)
    return curriculum

health = check_backend()
scene = run_scene_generation(SCENE_PROMPT)
taskset = create_taskset(scene)
training = create_training_run(taskset)
curriculum = create_curriculum_run(scene, taskset, training)

summary = {
    "runId": RUN_ID,
    "outputDir": str(OUT_DIR),
    "health": health,
    "sceneId": scene["id"],
    "taskSetId": taskset["id"],
    "trainingRunId": training["id"],
    "curriculumRunId": curriculum["id"],
    "status": "complete_artifact_pipeline",
}
write_json("summary.json", summary)
print("\nFull artifact pipeline complete.")
print("Output directory:", OUT_DIR)
